# Exercise 04: Incremental Modeling

In this exercise you'll implement an **incremental model** — one of the most important patterns in modern data engineering.

---

### Full refresh vs. incremental

| Strategy | What it does | When to use |
|---|---|---|
| **Full refresh** | Drop and rebuild the entire table every run | Small tables, or when data changes unpredictably |
| **Incremental** | Process only new or changed records, merge into the target | Large tables, event data, daily status updates |

At scale, rebuilding a 500M-row table from scratch every hour isn't feasible. Incremental models process only what's new or changed since the last run.

---

### The two concepts you'll use

**1. Lookback window** — filter the source to only records updated in the last N days.
A small overlap (e.g. 3 days) is used to catch **late-arriving data**: records that were updated days ago in the source but only appeared in today's snapshot.

**2. `MERGE INTO`** — a single SQL statement that does two things at once:
- Record already exists in the target → **UPDATE** it
- Record is brand new → **INSERT** it

This is also called an **upsert**.

## The Scenario

You maintain a `fct_orders` table in your warehouse. Every day, the source system sends a **full snapshot** of all orders — but only some records have actually changed.

**Today's date: `2024-04-24`**

| File | What it represents |
|---|---|
| `orders_initial.csv` | The 10 orders currently in your warehouse |
| `orders_updated.csv` | The full source snapshot arriving today (13 rows) |

The `updated_at` column records when each order was last modified in the source system. This is the key to identifying what actually needs processing.

**Changes in today's snapshot:**

| Order | Change | `updated_at` |
|---|---|---|
| 1005 | `shipped` → `completed` | 2024-04-22 |
| 1006 | `shipped` → `completed` | 2024-04-23 |
| 1007 | `placed` → `shipped` | 2024-04-22 |
| 1008 | `placed` → `shipped` | 2024-04-23 |
| 1009 | `placed` → `cancelled` | 2024-04-21 ← late-arriving data |
| 1011 | New order | 2024-04-22 |
| 1012 | New order | 2024-04-23 |
| 1013 | New order | 2024-04-24 |

## Setup

In [ ]:
import duckdb

con = duckdb.connect()
print('Connected! DuckDB version:', duckdb.__version__)

---
## Step 1: Build the initial warehouse table

Load `orders_initial.csv` to create your starting `fct_orders` table.
This represents the current state of your warehouse before today's run.

In [ ]:
con.execute('''
    CREATE OR REPLACE TABLE fct_orders AS
    SELECT * FROM read_csv_auto('../data/incremental/orders_initial.csv')
''')

print('fct_orders loaded with', con.execute('SELECT COUNT(*) FROM fct_orders').fetchone()[0], 'rows.')

In [ ]:
# Current state of the warehouse — 10 orders, mix of statuses
con.execute('SELECT * FROM fct_orders ORDER BY order_id').df()

---
## Step 2: The source snapshot arrives

Load `orders_updated.csv` into a staging table called `orders_source`.
This is the full snapshot from the source system — all 13 orders, including updates and new records.

In [ ]:
con.execute('''
    CREATE OR REPLACE TABLE orders_source AS
    SELECT * FROM read_csv_auto('../data/incremental/orders_updated.csv')
''')

print('orders_source loaded with', con.execute('SELECT COUNT(*) FROM orders_source').fetchone()[0], 'rows.')

In [ ]:
# The source snapshot — notice the updated statuses and 3 new orders at the bottom
con.execute('SELECT * FROM orders_source ORDER BY order_id').df()

---
## Step 3: What changed?

Before writing any logic, let's see exactly what the source snapshot contains that differs from the warehouse.
This query finds status changes and new orders — it's useful for understanding the data, not something you'd run in production.

In [ ]:
con.execute('''
    -- Orders that changed status
    SELECT
        s.order_id,
        t.status    AS warehouse_status,
        s.status    AS source_status,
        s.updated_at
    FROM orders_source s
    JOIN fct_orders t ON s.order_id = t.order_id
    WHERE s.status != t.status

    UNION ALL

    -- New orders not yet in the warehouse
    SELECT
        s.order_id,
        NULL        AS warehouse_status,
        s.status    AS source_status,
        s.updated_at
    FROM orders_source s
    LEFT JOIN fct_orders t ON s.order_id = t.order_id
    WHERE t.order_id IS NULL

    ORDER BY order_id
''').df()

---
## Step 4: The lookback window

The source snapshot has 13 rows, but only 8 actually need processing.
Orders 1001–1004 and 1010 haven't changed — their `updated_at` is old.

Rather than comparing every row, we use a **lookback window**: only process records where `updated_at` falls within the last 3 days.

```
Run date:    2024-04-24
Lookback:    3 days
Cut-off:     2024-04-21

Process records where:  updated_at >= DATE '2024-04-24' - INTERVAL 3 DAY
```

Order 1009 (`updated_at = 2024-04-21`) is exactly at the boundary — the 3-day overlap is what catches **late-arriving data** like this.

**Task:** Create a table called `incremental_batch` containing only the records from `orders_source` that fall within the 3-day lookback window.

In [ ]:
con.execute('''
    CREATE OR REPLACE TABLE incremental_batch AS
    SELECT *
    FROM orders_source
    WHERE updated_at >= DATE '2024-04-24' - INTERVAL 3 DAY
''')

In [ ]:
# ✅ Should have 8 rows: 5 status changes + 3 new orders
print('Rows in incremental batch:', con.execute('SELECT COUNT(*) FROM incremental_batch').fetchone()[0])
con.execute('SELECT * FROM incremental_batch ORDER BY order_id').df()

---
## Challenge: `MERGE INTO`

Now apply the incremental batch to `fct_orders` using a single `MERGE INTO` statement.

The `MERGE INTO` statement matches rows between the source and target using the **primary key** (`order_id`) and handles both cases in one go:

```sql
MERGE INTO target AS t
USING source AS s
ON t.primary_key = s.primary_key          -- match on PK
WHEN MATCHED THEN
    UPDATE SET col = s.col, ...           -- record exists: update it
WHEN NOT MATCHED THEN
    INSERT (col1, col2, ...)              -- record is new: insert it
    VALUES (s.col1, s.col2, ...);
```

**What to update when matched:** `status`, `amount`, `updated_at`

**What to insert when not matched:** all columns — `order_id`, `customer_id`, `order_date`, `status`, `amount`, `updated_at`

In [ ]:
con.execute('''
    MERGE INTO fct_orders AS target
    USING incremental_batch AS source
    ON target.order_id = source.order_id

    WHEN MATCHED THEN
        UPDATE SET
            status     = source.status,
            amount     = source.amount,
            updated_at = source.updated_at

    WHEN NOT MATCHED THEN
        INSERT (order_id, customer_id, order_date, status, amount, updated_at)
        VALUES (source.order_id, source.customer_id, source.order_date, source.status, source.amount, source.updated_at)
''')

---
## Step 6: Verify the results

In [ ]:
# Final row count — should be 13 (10 original + 3 new)
print('Final row count:', con.execute('SELECT COUNT(*) FROM fct_orders').fetchone()[0], '(expected 13)')

con.execute('SELECT * FROM fct_orders ORDER BY order_id').df()

In [ ]:
# Show which orders were updated (status now differs from what was in orders_initial)
con.execute('''
    CREATE OR REPLACE TABLE orders_original AS
    SELECT * FROM read_csv_auto('../data/incremental/orders_initial.csv')
''')

con.execute('''
    SELECT
        f.order_id,
        o.status    AS before,
        f.status    AS after,
        f.updated_at
    FROM fct_orders f
    LEFT JOIN orders_original o ON f.order_id = o.order_id
    WHERE o.status IS NULL OR f.status != o.status
    ORDER BY f.order_id
''').df()

---
## Summary

| Concept | What it does |
|---|---|
| `updated_at` column | Records when the source row last changed — the engine of incremental logic |
| Lookback window | Filters the source to only recently changed records; the overlap catches late-arriving data |
| `MERGE INTO ... ON primary_key` | Updates existing rows and inserts new ones in a single statement |
| `WHEN MATCHED THEN UPDATE` | Applies changes to records that already exist in the target |
| `WHEN NOT MATCHED THEN INSERT` | Adds brand-new records to the target |

**In dbt, the `incremental` materialization automates exactly this pattern** — but now you understand the SQL it generates under the hood.